# All-model VAD benchmark

This notebook loads existing checkpoints and performs inference only. It compares the classical VAD, trained CRNN, trained Kaggle CNN-BiLSTM, default Silero, and Silero using the previously selected dev configuration (threshold=0.35, minimum speech=400 ms, minimum silence=400 ms). No 27-way sweep or neural retraining is performed.


Run cells from top to bottom. Outputs are saved under `results/all_vad_benchmark/`.


## 1. Setup and paths


In [25]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
)


ROOT = Path.cwd()
if not (ROOT / 'features').exists():
    raise FileNotFoundError('Run this notebook with the project root as the working directory.')
FEATURES_ROOT = ROOT / "features"
AUDIO_ROOT = ROOT / "LibriParty" / "dataset"
CHECKPOINT_ROOT = ROOT / "checkpoints"
KAGGLE_ROOT = ROOT / "Kaggle_outputs"
RESULTS_ROOT = ROOT / "results" / "all_vad_benchmark"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 16_000
HOP_LENGTH = 160
HOP_SEC = HOP_LENGTH / SAMPLE_RATE
N_MELS = 40
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")


## 2. Session loading and configuration


In [26]:
def load_sessions(split: str) -> list[dict]:
    """Load cached features and their ground-truth labels for one split."""
    sessions = []
    for path in sorted((FEATURES_ROOT / split).glob("*_features.npz")):
        data = np.load(path)
        name = path.stem.replace("_mixture_features", "")
        audio_path = AUDIO_ROOT / split / name / f"{name}_mixture.wav"
        sessions.append(
            {
                "name": name,
                "energy_db": data["energy_db"].astype(np.float32),
                "zcr": data["zcr"].astype(np.float32),
                "log_mel_norm": data["log_mel_norm"].astype(np.float32),
                "frame_labels": data["frame_labels"].astype(int),
                "frame_times": data["frame_times"].astype(np.float32),
                "audio_path": audio_path,
            }
        )
    if not sessions:
        raise FileNotFoundError(f"No cached feature files found in {FEATURES_ROOT / split}")
    return sessions


with (CHECKPOINT_ROOT / "classical_vad_params.json").open() as file:
    CLASSICAL_PARAMS = json.load(file)


## 3. Existing CRNN architecture


In [27]:
class CRNNVAD(nn.Module):
    """Architecture used by vad_model_training.ipynb."""

    def __init__(self, n_mels: int = N_MELS, conv_channels: int = 32, gru_hidden: int = 64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, conv_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(conv_channels),
            nn.ReLU(),
        )
        self.freq_pool = nn.AdaptiveAvgPool2d((1, None))
        self.gru = nn.GRU(conv_channels, gru_hidden, batch_first=True, bidirectional=True)
        self.head = nn.Linear(gru_hidden * 2, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        hidden = self.freq_pool(self.conv(x)).squeeze(2).transpose(1, 2)
        output, _ = self.gru(hidden)
        return self.head(output).squeeze(-1)


## 4. Kaggle CNN-BiLSTM architecture


In [28]:
class CNNBiLSTM(nn.Module):
    """Architecture used by cnn_lstm_vad_kaggle.ipynb."""

    def __init__(
        self,
        channels: int = 64,
        hidden: int = 128,
        layers: int = 2,
        dropout: float = 0.25,
    ):
        super().__init__()
        self.frontend = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Dropout2d(dropout),
        )
        self.freq_pool = nn.AdaptiveAvgPool2d((1, None))
        self.rnn = nn.LSTM(
            channels,
            hidden,
            num_layers=layers,
            batch_first=True,
            dropout=dropout if layers > 1 else 0,
            bidirectional=True,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        hidden = self.freq_pool(self.frontend(x)).squeeze(2).transpose(1, 2)
        hidden, _ = self.rnn(hidden)
        return self.head(hidden).squeeze(-1)


## 5. Load trained checkpoints


In [29]:
def load_neural_model(model: nn.Module, checkpoint: Path) -> nn.Module:
    payload = torch.load(checkpoint, map_location=DEVICE)
    state = payload.get("model_state", payload) if isinstance(payload, dict) else payload
    model.load_state_dict(state)
    return model.to(DEVICE).eval()


crnn = load_neural_model(CRNNVAD(), CHECKPOINT_ROOT / "crnn_vad_best.pt")
cnn_bilstm = load_neural_model(CNNBiLSTM(), KAGGLE_ROOT / "cnn_bilstm_vad_best.pt")


## 6. Classical and neural inference


In [30]:
def classical_predict(session: dict) -> np.ndarray:
    energy = session["energy_db"]
    zcr = session["zcr"]
    core = energy > CLASSICAL_PARAMS["t_energy"]
    rescue = (
        (energy > CLASSICAL_PARAMS["t_energy"] - CLASSICAL_PARAMS["margin_db"])
        & (zcr >= CLASSICAL_PARAMS["zcr_low"])
        & (zcr <= CLASSICAL_PARAMS["zcr_high"])
    )
    return (core | rescue).astype(int)


@torch.no_grad()
def neural_probabilities(model: nn.Module, session: dict) -> np.ndarray:
    features = torch.from_numpy(session["log_mel_norm"])[None, None].float().to(DEVICE)
    logits = model(features).squeeze(0).cpu().numpy()
    return 1.0 / (1.0 + np.exp(-logits))


## 7. Silero loading and inference


In [31]:
def load_silero():
    try:
        model, utils = torch.hub.load(
            "snakers4/silero-vad",
            "silero_vad",
            trust_repo=True,
            verbose=False,
        )
        return model.to("cpu").eval(), utils[0]
    except Exception as exc:
        print(f"Silero unavailable: {type(exc).__name__}: {exc}")
        return None, None


SILERO_MODEL, GET_SPEECH_TIMESTAMPS = load_silero()


def silero_predict(
    session: dict,
    threshold: float = 0.5,
    min_speech_ms: int = 0,
    min_silence_ms: int = 0,
) -> np.ndarray | None:
    if SILERO_MODEL is None:
        return None
    audio, _ = librosa.load(session["audio_path"], sr=SAMPLE_RATE, mono=True)
    timestamps = GET_SPEECH_TIMESTAMPS(
        torch.from_numpy(audio).float(),
        SILERO_MODEL,
        sampling_rate=SAMPLE_RATE,
        threshold=threshold,
        min_speech_duration_ms=min_speech_ms,
        min_silence_duration_ms=min_silence_ms,
        return_seconds=True,
    )
    prediction = np.zeros(len(session["frame_times"]), dtype=int)
    for segment in timestamps:
        mask = (session["frame_times"] >= segment["start"]) & (
            session["frame_times"] < segment["end"]
        )
        prediction[mask] = 1
    return prediction


## 8. Pause-event metrics


In [32]:
def binary_segments(labels: np.ndarray, times: np.ndarray) -> list[tuple[float, float]]:
    """Convert frame labels into half-open time intervals."""
    labels = np.asarray(labels).astype(bool)
    if not labels.any():
        return []
    starts = np.flatnonzero(labels & ~np.r_[False, labels[:-1]])
    ends = np.flatnonzero(labels & ~np.r_[labels[1:], False])
    return [(float(times[start]), float(times[end] + HOP_SEC)) for start, end in zip(starts, ends)]


def interval_iou(first: tuple[float, float], second: tuple[float, float]) -> float:
    intersection = max(0.0, min(first[1], second[1]) - max(first[0], second[0]))
    union = max(first[1], second[1]) - min(first[0], second[0])
    return intersection / union if union else 0.0


def pause_metrics(
    prediction: np.ndarray,
    truth: np.ndarray,
    times: np.ndarray,
    iou_threshold: float = 0.5,
    tolerance: float = 0.25,
) -> dict:
    """Measure pause events, where pause is the negative/non-speech class."""
    predicted = binary_segments(prediction == 0, times)
    actual = binary_segments(truth == 0, times)
    matches = sorted(
        (interval_iou(predicted_i, actual_i), predicted_i, actual_i)
        for predicted_i in predicted
        for actual_i in actual
        if interval_iou(predicted_i, actual_i) >= iou_threshold
    )
    used_pred, used_actual = set(), set()
    selected = []
    for score, predicted_i, actual_i in sorted(matches, reverse=True):
        if predicted_i not in used_pred and actual_i not in used_actual:
            used_pred.add(predicted_i)
            used_actual.add(actual_i)
            selected.append((score, predicted_i, actual_i))
    precision = len(selected) / len(predicted) if predicted else 0.0
    recall = len(selected) / len(actual) if actual else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    onset_errors = [abs(a[1][0] - a[2][0]) for a in selected]
    offset_errors = [abs(a[1][1] - a[2][1]) for a in selected]
    boundaries = [
        abs(a[1][0] - a[2][0]) <= tolerance or abs(a[1][1] - a[2][1]) <= tolerance
        for a in selected
    ]
    return {
        "actual_pauses": len(actual),
        "predicted_pauses": len(predicted),
        "matched_pauses": len(selected),
        "pause_precision": precision,
        "pause_recall": recall,
        "pause_f1": f1,
        "mean_pause_iou": float(np.mean([a[0] for a in selected])) if selected else 0.0,
        "mean_onset_error_sec": float(np.mean(onset_errors)) if onset_errors else 0.0,
        "mean_offset_error_sec": float(np.mean(offset_errors)) if offset_errors else 0.0,
        "boundary_within_250ms": float(np.mean(boundaries)) if boundaries else 0.0,
    }


## 9. Model scoring


In [33]:
def score_model(model_name: str, predictions: dict[str, np.ndarray], sessions: list[dict]) -> pd.DataFrame:
    rows = []
    for session in sessions:
        prediction = predictions[session["name"]]
        truth = session["frame_labels"]
        precision, recall, f1, _ = precision_recall_fscore_support(
            truth, prediction, average="binary", zero_division=0
        )
        row = {
            "model": model_name,
            "session": session["name"],
            "precision": precision,
            "recall": recall,
            "frame_f1": f1,
            "accuracy": accuracy_score(truth, prediction),
            "balanced_accuracy": balanced_accuracy_score(truth, prediction),
        }
        row.update(pause_metrics(prediction, truth, session["frame_times"]))
        rows.append(row)
    return pd.DataFrame(rows)


def predict_fixed_models(sessions: list[dict]) -> dict[str, dict[str, np.ndarray]]:
    return {
        "Classical VAD": {s["name"]: classical_predict(s) for s in sessions},
        "Neural CRNN": {
            s["name"]: (neural_probabilities(crnn, s) >= 0.5).astype(int) for s in sessions
        },
        "CNN-BiLSTM (Kaggle)": {
            s["name"]: (neural_probabilities(cnn_bilstm, s) >= 0.4).astype(int)
            for s in sessions
        },
    }


## 10. Silero previously selected configuration


In [34]:
def get_silero_dev_config() -> dict | None:
    if SILERO_MODEL is None:
        return None
    config = {"threshold": 0.35, "min_speech_ms": 400, "min_silence_ms": 400}
    print("Using previously selected Silero dev configuration:", config)
    return config


## 11. Comparison plots and qualitative timeline


In [35]:
def plot_outputs(
    summary: pd.DataFrame,
    per_session: pd.DataFrame,
    prediction_sets: dict[str, dict[str, np.ndarray]],
    eval_sessions: list[dict],
) -> None:
    display_names = {
        "Classical VAD": "Classical",
        "Neural CRNN": "CRNN",
        "CNN-BiLSTM (Kaggle)": "CNN-LSTM",
        "Silero VAD (default)": "Silero",
        "Silero VAD (dev-tuned)": "Silero*",
    }
    plot_summary = summary.rename(index=display_names).copy()
    plot_per_session = per_session.assign(model=per_session["model"].map(display_names))
    model_order = [name for name in ["Classical", "CRNN", "CNN-LSTM", "Silero", "Silero*"] if name in plot_summary.index]

    plot_metrics = plot_summary.reset_index().melt(
        id_vars="model",
        value_vars=["frame_f1", "pause_f1", "boundary_within_250ms"],
        var_name="metric",
        value_name="score",
    )
    metric_names = {
        "frame_f1": "Frame F1",
        "pause_f1": "Pause F1",
        "boundary_within_250ms": "Boundary ≤250 ms",
    }
    plot_metrics["metric"] = plot_metrics["metric"].map(metric_names)

    fig, axis = plt.subplots(figsize=(12, 6.5), constrained_layout=True)
    sns.barplot(
        plot_metrics,
        x="metric",
        y="score",
        hue="model",
        hue_order=model_order,
        ax=axis,
        palette="Set2",
        errorbar=None,
    )
    axis.set_ylim(0, 1.08)
    axis.set_title("Held-out LibriParty performance", pad=16, weight="bold")
    axis.set_xlabel("")
    axis.set_ylabel("Macro-average score")
    axis.legend(
        title="Model",
        ncol=3,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.12),
        frameon=False,
    )
    for container in axis.containers:
        axis.bar_label(container, fmt="%.2f", padding=3, fontsize=8)
    fig.savefig(RESULTS_ROOT / "all_vad_model_comparison.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), constrained_layout=True)
    for axis, metric, title in zip(
        axes,
        ["frame_f1", "pause_f1"],
        ["Frame F1 by session", "Pause-event F1 by session"],
    ):
        sns.boxplot(
            plot_per_session,
            x="model",
            y=metric,
            order=model_order,
            hue="model",
            hue_order=model_order,
            legend=False,
            palette="Set2",
            ax=axis,
        )
        sns.stripplot(
            plot_per_session,
            x="model",
            y=metric,
            order=model_order,
            color="black",
            alpha=0.35,
            size=3,
            jitter=0.18,
            ax=axis,
        )
        axis.set_title(title, pad=12, weight="bold")
        axis.set_ylim(0, 1.05)
        axis.set_xlabel("")
        axis.set_ylabel(metric.replace("_", " ").title())
        axis.tick_params(axis="x", rotation=0)
    fig.savefig(RESULTS_ROOT / "all_vad_model_stability.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    example = eval_sessions[0]
    fig, axes = plt.subplots(
        len(prediction_sets) + 1,
        1,
        figsize=(16, 1.9 * (len(prediction_sets) + 1)),
        sharex=True,
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes)
    axes[0].plot(
        example["frame_times"],
        example["frame_labels"],
        drawstyle="steps-post",
        color="black",
        linewidth=1.2,
    )
    axes[0].set_title(f"Qualitative VAD timeline — {example['name']}", pad=10, weight="bold")
    axes[0].set_ylabel("Truth", rotation=0, ha="right", va="center", labelpad=22)
    axes[0].set_ylim(-0.1, 1.1)
    for axis, (model_name, predictions) in zip(axes[1:], prediction_sets.items()):
        axis.plot(
            example["frame_times"],
            predictions[example["name"]],
            drawstyle="steps-post",
            linewidth=1.0,
        )
        axis.set_ylabel(
            display_names.get(model_name, model_name),
            rotation=0,
            ha="right",
            va="center",
            labelpad=22,
        )
        axis.set_ylim(-0.1, 1.1)
        axis.grid(True, axis="y", alpha=0.3)
    axes[-1].set_xlabel("Time (seconds)")
    fig.savefig(RESULTS_ROOT / "vad_qualitative_timeline_all_models.png", dpi=200, bbox_inches="tight")
    plt.close(fig)



## 12. Run benchmark and export results


In [36]:
def main() -> None:
    print("Device:", DEVICE)
    print("Loading cached sessions...")
    dev_sessions = load_sessions("dev")
    eval_sessions = load_sessions("eval")
    print(f"Dev sessions: {len(dev_sessions)} | Eval sessions: {len(eval_sessions)}")

    prediction_sets = predict_fixed_models(eval_sessions)
    if SILERO_MODEL is not None:
        prediction_sets["Silero VAD (default)"] = {
            s["name"]: silero_predict(s) for s in eval_sessions
        }
    silero_config = get_silero_dev_config()
    if silero_config is not None:
        prediction_sets["Silero VAD (dev-tuned)"] = {
            s["name"]: silero_predict(
                s,
                float(silero_config["threshold"]),
                int(silero_config["min_speech_ms"]),
                int(silero_config["min_silence_ms"]),
            )
            for s in eval_sessions
        }

    per_session = pd.concat(
        [score_model(name, predictions, eval_sessions) for name, predictions in prediction_sets.items()],
        ignore_index=True,
    )
    metric_columns = [
        "precision",
        "recall",
        "frame_f1",
        "accuracy",
        "balanced_accuracy",
        "pause_precision",
        "pause_recall",
        "pause_f1",
        "mean_pause_iou",
        "mean_onset_error_sec",
        "mean_offset_error_sec",
        "boundary_within_250ms",
    ]
    summary = per_session.groupby("model")[metric_columns].mean().sort_values(
        "frame_f1", ascending=False
    )
    print("\nHeld-out eval comparison (macro-average across sessions):")
    print(summary.round(4).to_string())

    per_session.to_csv(RESULTS_ROOT / "vad_benchmark_per_session.csv", index=False)
    summary.to_csv(RESULTS_ROOT / "vad_benchmark_summary.csv")
    plot_outputs(summary, per_session, prediction_sets, eval_sessions)
    payload = {
        "device": str(DEVICE),
        "eval_sessions": len(eval_sessions),
        "frame_hop_ms": HOP_SEC * 1000,
        "silero_dev_config": silero_config,
        "models": summary.round(6).to_dict(orient="index"),
        "checkpoints": {
            "crnn": str(CHECKPOINT_ROOT / "crnn_vad_best.pt"),
            "cnn_bilstm": str(KAGGLE_ROOT / "cnn_bilstm_vad_best.pt"),
        },
    }
    with (RESULTS_ROOT / "vad_benchmark_summary.json").open("w") as file:
        json.dump(payload, file, indent=2)

    print("\nSaved benchmark outputs:")
    for path in sorted(RESULTS_ROOT.iterdir()):
        print(" ", path)


main()


Device: cpu
Loading cached sessions...
Dev sessions: 50 | Eval sessions: 50
Using previously selected Silero dev configuration: {'threshold': 0.35, 'min_speech_ms': 400, 'min_silence_ms': 400}

Held-out eval comparison (macro-average across sessions):
                        precision  recall  frame_f1  accuracy  balanced_accuracy  pause_precision  pause_recall  pause_f1  mean_pause_iou  mean_onset_error_sec  mean_offset_error_sec  boundary_within_250ms
model                                                                                                                                                                                                        
CNN-BiLSTM (Kaggle)        0.9617  0.9678    0.9639    0.9680             0.9686           0.8120        0.8878    0.8425          0.9275                0.3809                 0.2463                 0.9557
Silero VAD (dev-tuned)     0.9983  0.8178    0.8925    0.9154             0.9083           0.6238        0.8445    0.7076         